# Step 0 · Validation of the SBERT filter on PROD

Measures how good the final dataset is, on the **PROD** run: **precision** (share of the kept families that are drones) and **recall** (share of the drones found by the search that the filter keeps).

- **Input:** `df_sbert_scored.parquet`, written by the data preparation notebook (part A) and scored on Colab (`1c-SbertScoring_Colab.ipynb`). The kept/rejected split is rebuilt here from `TAU_FILTER`.
- **Settings read from the data preparation notebook:** `DRONE_DESCRIPTION` (the sector definition given to the LLM) and `TAU_FILTER`, so the two notebooks cannot drift apart.
- **Method** (`2-DataValidation`): random families from the kept and from the rejected group (seed 42), one abstract per family, labelled by gpt-4.1-mini against `DRONE_DESCRIPTION`; `incerto` answers are labelled by hand (section 3); precision and recall with 95% Wilson intervals.
- **Cost:** about 5 cents per 100 LLM calls; families already labelled with the same definition are read from the cache.
- **Output:** `validation/prod_filter_validation_v2.csv` (numbers only) and the hand labels in `validation/review_<run>_*.csv`. Section 5 removes titles and abstracts from the review files before they are committed.

History: the first PROD validation (10 Nov; earlier definition, threshold 0.45, abstracts only) gave precision 0.88 (95% CI 0.80–0.93) and recall 0.94; its files are `prod_filter_validation.csv` and `review_filtro_sbert_prod_*.csv`, left unchanged.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

SCORED_FILE = Path("/home/jovyan/progetto/df_sbert_scored.parquet")   # written on Colab (1c)
N_KEPT = 100                          # random families checked among the kept ones
N_REJECTED = 100                      # and among the rejected ones
RUN_NAME = "Filtro SBERT PROD v2"     # review files: validation/review_filtro_sbert_prod_v2_*.csv
RESULT_FILE = "prod_filter_validation_v2.csv"   # the first PROD run (10 Nov) stays in prod_filter_validation.csv

# DRONE_DESCRIPTION and TAU_FILTER: read from the data preparation notebook (single source).
candidates = [p for p in sorted(Path(".").glob("1*-DataPreparation.ipynb"))
              if re.fullmatch(r"1[a-z]?-DataPreparation\.ipynb", p.name)]
assert len(candidates) == 1, (f"Expected one data preparation notebook in Step0, found {[p.name for p in candidates]}: "
                              "set PREP_NOTEBOOK = Path('<name>') by hand")
PREP_NOTEBOOK = candidates[0]
prep_code = "\n".join("".join(c["source"])
                      for c in json.loads(PREP_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
                      if c["cell_type"] == "code")
DRONE_DESCRIPTION = re.search(r'DRONE_DESCRIPTION = """(.*?)"""\.strip\(\)', prep_code, re.S).group(1).strip()
TAU_FILTER = float(re.search(r"^TAU_FILTER = ([0-9.]+)", prep_code, re.M).group(1))

print(f"Settings from {PREP_NOTEBOOK.name}: TAU_FILTER = {TAU_FILTER}")
print("Definition starts with:", DRONE_DESCRIPTION.splitlines()[0])

## 1. Kept and rejected families

A family is **kept** if at least one of its applications reaches the threshold; it is **rejected** if none does.

In [ ]:
scored = pd.read_parquet(SCORED_FILE)
scored = scored.dropna(subset=["docdb_family_id"]).copy()
scored["docdb_family_id"] = scored["docdb_family_id"].astype("int64")
scored = scored[scored["docdb_family_id"] > 0]

kept = scored[scored["sbert_similarity"] >= TAU_FILTER].copy()
kept_ids = set(kept["docdb_family_id"])
rejected = scored[(scored["sbert_similarity"] < TAU_FILTER)
                  & ~scored["docdb_family_id"].isin(kept_ids)].copy()

assert kept_ids.isdisjoint(set(rejected["docdb_family_id"]))
print(f"Applications scored: {len(scored):,} | kept: {len(kept):,} ({len(kept) / len(scored):.1%})")
print(f"Families kept: {kept['docdb_family_id'].nunique():,} | rejected: {rejected['docdb_family_id'].nunique():,}")

## 2. LLM labels

In [ ]:
%pip install -q langchain-core langchain-openai pydantic
%run -i ./2-DataValidation.ipynb

pid = prompt_id(DRONE_DESCRIPTION)
already = sum(1 for (family, p) in load_llm_cache() if p == pid)
print(f"Prompt id {pid}: {already} families already labelled with this definition (reused from the cache)")

validation_kept = validate_sample(kept, n=N_KEPT)
validation_rejected = validate_sample(rejected, n=N_REJECTED)

prod_metrics = evaluate_selection(
    validation_selected=validation_kept,
    validation_rejected=validation_rejected,
    df_selected=kept,
    df_rejected=rejected,
    name=RUN_NAME,
)
display(prod_metrics.to_frame().round(3))

## 3. Hand labels for the `incerto` cases

The first cell shows the families the LLM could not decide. Read them, fill `LABELS` in the second cell (`"pertinente"` or `"non_pertinente"`, with the same definition as the LLM: `DRONE_DESCRIPTION`), run it, then **Restart Kernel and Run All**: the LLM is not called again.

In [ ]:
slug = re.sub(r"[^a-z0-9]+", "_", RUN_NAME.lower()).strip("_")
REVIEW_FILES = {tag: VALIDATION_DIR / f"review_{slug}_{tag}.csv" for tag in ("selezionati", "scartati")}

for tag, path in REVIEW_FILES.items():
    if not path.exists():
        continue
    review = pd.read_csv(path, keep_default_na=False)
    group = "KEPT" if tag == "selezionati" else "REJECTED"
    for _, r in review.iterrows():
        print(f"\n[{r['docdb_family_id']}] ({group}) {r.get('appln_title', '')}\n  {r.get('appln_abstract', '')}\n"
              f"  LLM: {r['reason']}\n  hand label: {r['human_label'] or '-'}")

In [ ]:
LABELS = {
    # docdb_family_id: "pertinente" or "non_pertinente"
}

assert set(LABELS.values()) <= {"pertinente", "non_pertinente"}, "Use only pertinente / non_pertinente"
for path in REVIEW_FILES.values():
    if path.exists():
        review = pd.read_csv(path, keep_default_na=False)
        review["human_label"] = review["docdb_family_id"].map(LABELS).fillna(review["human_label"])
        review.to_csv(path, index=False)
        print(path.name, "| still empty:", int((review["human_label"].astype(str).str.strip() == "").sum()))

## 4. Results

- **precision:** share of the kept families that are drones; **estimated drone families kept** = kept families × precision.
- **relevant_share_in_rejected:** share of drones among the rejected families; **estimated drone families lost** = rejected families × that share.
- **recall_within_pool:** drones kept / drones found by the search (the search itself can miss drones: this is recall *within* the search results).

Then the errors: non-drones among the kept families (they lower precision) and drones among the rejected ones (they lower recall). Shown here only (titles are not saved), to describe the errors in the appendix.

In [ ]:
if prod_metrics.get("to_check_by_hand", 0):
    print("Label the 'incerto' cases first (section 3), then re-run.")
else:
    summary = prod_metrics.astype(float)
    summary["estimated_drone_families_kept"] = summary["selected_families"] * summary["precision"]
    summary["estimated_drone_families_lost"] = (summary["rejected_families"]
                                                * summary["relevant_share_in_rejected"])
    summary = summary.round(4)
    summary["tau_filter"] = TAU_FILTER
    summary["prompt_id"] = pid
    summary.to_frame().T.to_csv(VALIDATION_DIR / RESULT_FILE, index=False)
    display(summary.to_frame())
    print("Saved:", VALIDATION_DIR / RESULT_FILE)

    for title, sample, tag, wrong in [
        ("Kept but not drones", validation_kept, "selezionati", "non_pertinente"),
        ("Rejected but drones", validation_rejected, "scartati", "pertinente"),
    ]:
        labels = _resolved_labels(sample, REVIEW_FILES[tag])
        errors = sample.reset_index(drop=True)[labels.eq(wrong)]
        print(f"\n{title}: {len(errors)} of {len(sample)}")
        display(errors[["docdb_family_id", "sbert_similarity", "appln_title", "reason"]])

## 5. Before committing: remove the patent texts

Run this cell only when all the hand labels are filled. It keeps a backup with the texts outside the repository and leaves in the review files only the family id, the LLM answer, its reason and the hand label (the notebook needs nothing else).

In [ ]:
BACKUP_DIR = Path("/home/jovyan/progetto/validation_texts_backup")
KEEP = ["docdb_family_id", "llm_label", "reason", "human_label"]

BACKUP_DIR.mkdir(parents=True, exist_ok=True)
for path in REVIEW_FILES.values():
    if not path.exists():
        continue
    review = pd.read_csv(path, keep_default_na=False)
    if (review["human_label"].astype(str).str.strip() == "").any():
        print(f"{path.name}: some hand labels are still empty, nothing changed")
        continue
    review.to_csv(BACKUP_DIR / path.name, index=False)
    review[KEEP].to_csv(path, index=False)
    print(f"{path.name}: {len(review)} rows, columns now {KEEP} (backup in {BACKUP_DIR})")